# ch06 数据构造与标准化（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv`（A 居民型 / B 工业型）+ `data/pv_power.csv`（3 特征）。
> `window=72` / `horizon=24`，与 ch05 对齐。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 分组清洗 + 多特征矩阵 | 3 | §6.1 |
| 2 | 手写 `WindowDataset` | 3 | §6.1 |
| 3 | 跨界窗口 vs 分组切窗 | 2 | §6.1.1 |
| 4 | 跨界污染的量化证据 | 3 | §6.1.1 |
| 5 | `DataLoader` 形状 + `shuffle` 边界 | 4 | §6.2.1 |
| 6 | 手写 `fit` / `apply`（z-score + min-max） | 4 | §6.3 |
| 7 | z-score 防泄漏：训练段 vs 全量 | 2 | §6.3.1 |
| 8 | min-max 防泄漏：训练段 vs 全量 | 2 | §6.3.1 |
| 9 | 手写 `TimeSeriesSplit` | 1 | §6.4 |
| 10 | 与 sklearn 对账 + `gap` 语义 | 2 | §6.4.1 |
| 11 | 滑窗 vs 扩展窗 | 1 | §6.5.1 |
| 12 | 两张图：折切分示意 + 折 MAE 对照 | 1 | §6.5 |

**合计 28 个挖空。**

> ⚠️ 第 6 题里手写的 4 个函数**在练习版里函数体会变成 `____`**，
> 所以它们在被调用时会报错 —— 这没关系：练习版在**第一个** `____` 处就停了。
> 你要做的是照着提示把它们补全，然后对照第 7、8 题的 `assert`。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"
PV_CSV = DATA / "pv_power.csv"

torch.set_num_threads(4)     # 固定线程数：结果可复现的前提之一

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
np.set_printoptions(precision=4, suppress=True)

print("torch", torch.__version__, "| sklearn 已就绪 | numpy", np.__version__)

In [ ]:
WINDOW, HORIZON = 72, 24     # 输入窗口 3 天，预测未来 1 天
SEED = 0

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
pv = pd.read_csv(PV_CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
pv = pv.drop_duplicates().sort_values("时间戳").reset_index(drop=True)

print("负荷表：原始 %d 行 → 去重后 %d 行 | 台区 %s"
      % (len(raw), len(df), sorted(df["台区编号"].unique())))
print("光伏表：原始 %d 行 → 去重后 %d 行 | 电站 %s"
      % (len(pd.read_csv(PV_CSV, encoding="utf-8-sig")), len(pv),
         sorted(pv["电站编号"].unique())))
print("光伏列", list(pv.columns))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的工具函数两版都有，你只填 @@todo 块里的内容


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值（与 ch01 / ch05 同一口径）。"""
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


def make_xy(arr, sel):
    """把一条序列按给定的起点集合切成 (输入窗口, 未来目标) 两块。

    返回的是 float32 numpy 数组：X 形状 (n, WINDOW)、y 形状 (n, HORIZON)。
    这里刻意**只吃一条序列** —— 想混两条序列，得自己先拼，拼错就是 §6.1.1 的坑。
    """
    x = np.stack([arr[i: i + WINDOW] for i in sel]).astype(np.float32)
    y = np.stack([arr[i + WINDOW: i + WINDOW + HORIZON] for i in sel]).astype(np.float32)
    return x, y


def ridge_mae(Xtr, Ytr, Xte, Yte, alpha=1.0):
    """多输出 Ridge + MAE。

    ⚠️ 这里 X 的标准化参数**只用训练段**拟合 —— 这就是本章 §6.3 的正确做法本身。
    """
    mu, sd = float(Xtr.mean()), float(Xtr.std())
    model = Ridge(alpha=alpha).fit((Xtr - mu) / sd, Ytr)
    return float(mean_absolute_error(Yte.ravel(), model.predict((Xte - mu) / sd).ravel()))

## 任务 1：按实体清洗 + 拼出多特征矩阵（§6.1）

In [ ]:
def series_of(name):
    """从一个台区名取清洗后的负荷序列（8760 点，无缺失）。"""
    fr = df[df["台区编号"] == name].set_index("时间戳").sort_index()
    return clean_series(fr["负荷值"].asfreq("h")).values.astype(np.float64)


S_A, S_B = series_of("STATION_A_01"), series_of("STATION_B_02")

pv_fr = pv.set_index("时间戳").sort_index()

act = pv_fr["实际功率"].asfreq("h").clip(lower=0).interpolate()
PV3 = np.stack([act.values,
                pv_fr["理论功率"].asfreq("h").interpolate().values,
                pv_fr["辐照度"].asfreq("h").interpolate().values], axis=1).astype(np.float32)

print("形状：S_A %s | S_B %s | PV3 %s（T 个时刻 × F 个特征）"
      % (S_A.shape, S_B.shape, PV3.shape))
print("A / B / PV实际功率 三条序列的尺度对照：")
print("  均值   %9.4f %9.4f %9.4f" % (S_A.mean(), S_B.mean(), PV3[:, 0].mean()))
print("  标准差 %9.4f %9.4f %9.4f  ← 相差 %.2f 倍"
      % (S_A.std(), S_B.std(), PV3[:, 0].std(), S_B.std() / S_A.std()))
print("PV 实际功率：清洗前 min %.2f（负值）→ 清洗后 min %.2f | max %.2f"
      % (pv_fr["实际功率"].min(), act.min(), act.max()))

# ---- 验收 ----
assert S_A.shape == (8760,) and S_B.shape == (8760,) and PV3.shape == (8760, 3)
assert abs(S_A.mean() - 460.201858) < 1e-4
assert abs(S_B.mean() - 473.850483) < 1e-4, "B 是工业型，均值与 A 接近但波动大得多"
assert abs(S_B.std() / S_A.std() - 3.895) < 1e-3, "B 的标准差是 A 的 3.9 倍"
assert float(PV3[:, 0].min()) == 0.0, "负值必须先 clip 掉"

## 任务 2：手写 `WindowDataset`（`window` / `horizon` / `stride` + 多特征）（§6.1）

In [ ]:
class WindowDataset(torch.utils.data.Dataset):
    """滑窗数据集：**每条序列各自独立切窗**，窗口绝不跨越两条序列的边界。

    series_list 里每个元素是一条序列：
      - 1-D (T,)   → 单变量（负荷），内部会 reshape 成 (T, 1)
      - 2-D (T, F) → 多特征（光伏：实际功率 / 理论功率 / 辐照度）
    统一成 (T, F) 之后，`__getitem__` 返回的 x 永远是 (window, F)、y 永远是 (horizon,)，
    **形状不带"是不是多变量"的分支** —— 下游模型只认 F 这一个数字。
    target_col 指定哪一列是预测目标。
    """

    def __init__(self, series_list, window, horizon, stride=1, target_col=0):
        self.window = window
        self.horizon = horizon
        self.target_col = target_col
        self.series = [np.asarray(a, dtype=np.float32).reshape(len(a), -1) for a in series_list]
        ends = [max(a.shape[0] - window - horizon + 1, 0) for a in self.series]
        self.index = [(k, i) for k, n in enumerate(ends) for i in range(0, n, stride)]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, j):
        k, i = self.index[j]
        arr = self.series[k]
        x = arr[i: i + self.window]
        y = arr[i + self.window: i + self.window + self.horizon, self.target_col]
        return torch.from_numpy(x), torch.from_numpy(y)


ds_ab = WindowDataset([S_A, S_B], WINDOW, HORIZON)
x0, y0 = ds_ab[0]
each = len(S_A) - WINDOW - HORIZON + 1
print("单条序列切出 %d 个样本；两条序列分组构造 → %d 个" % (each, len(ds_ab)))
print("第 0 个样本：x %s | y %s   ← 单变量时 F=1" % (tuple(x0.shape), tuple(y0.shape)))
print("索引表前 2 项 %s | 最后 2 项 %s" % (ds_ab.index[:2], ds_ab.index[-2:]))

ds_stride = WindowDataset([S_A, S_B], WINDOW, HORIZON, stride=4)
print("stride=4 → %d 个样本（%.1f%%）" % (len(ds_stride), 100 * len(ds_stride) / len(ds_ab)))

# ---- 验收 ----
assert len(ds_ab) == 17330, "两条序列各自切 8665 个，合计 17330"
assert tuple(x0.shape) == (72, 1) and tuple(y0.shape) == (24,)
assert ds_ab.index[0] == (0, 0) and ds_ab.index[-1] == (1, each - 1)
assert len(ds_stride) == 4334, "stride=4 → 每条 2167 个"
assert isinstance(x0, torch.Tensor)

## 任务 3：跨界切窗 vs 分组切窗 —— 数出 95 个假样本（§6.1.1）

In [ ]:
concat = np.concatenate([S_A, S_B])
CUT = len(S_A)                       # 8760：A 与 B 的接缝位置

ds_bad = WindowDataset([concat], WINDOW, HORIZON)

ds_ok = WindowDataset([S_A, S_B], WINDOW, HORIZON)

starts = np.array([i for _, i in ds_bad.index])
cross = starts[(starts < CUT) & (starts + WINDOW + HORIZON > CUT)]
xcross = starts[(starts < CUT) & (starts + WINDOW > CUT)]
ycross = starts[(starts + WINDOW <= CUT) & (starts + WINDOW + HORIZON > CUT)]

print("concat 构造 %d 个样本 | 分组构造 %d 个样本 | 多出来 %d 个"
      % (len(ds_bad), len(ds_ok), len(ds_bad) - len(ds_ok)))
print("跨界样本 %d 个（起点 %d ~ %d）" % (len(cross), cross.min(), cross.max()))
print("  其中「输入窗口跨边界」%d 个 | 「目标跨边界」%d 个" % (len(xcross), len(ycross)))
print("接缝两侧：A 的最后一小时（2025-12-31 23:00）负荷 %.2f" % S_A[-1])
print("          B 的第一小时（2025-01-01 00:00）负荷 %.2f  ← 时间倒退了 364 天" % S_B[0])

# ---- 验收 ----
assert len(ds_bad) == 17425 and len(ds_ok) == 17330
assert len(cross) == 95 and len(xcross) == 71 and len(ycross) == 24
assert CUT == 8760

## 任务 4：跨界污染有多严重 —— 干净模型在假样本上差 2.6 倍（§6.1.1）

In [ ]:
SEL_B = np.arange(len(S_B) - WINDOW - HORIZON + 1)
Xb_tr, Yb_tr = make_xy(S_B, SEL_B[:int(0.8 * len(S_B)) - WINDOW - HORIZON + 1])
Xb_te, Yb_te = make_xy(S_B, SEL_B[int(0.8 * len(S_B)):])

mae_clean = ridge_mae(Xb_tr, Yb_tr, Xb_te, Yb_te)

Xg, Yg = make_xy(concat, cross)
mae_cross = ridge_mae(Xb_tr, Yb_tr, Xg, Yg)

mae_mix = ridge_mae(np.concatenate([Xb_tr, Xg]), np.concatenate([Yb_tr, Yg]), Xb_te, Yb_te)

jump_t = [float(np.abs(np.diff(concat[i + WINDOW: i + WINDOW + HORIZON])).max()) for i in cross]
print("B 测试段 MAE（只用 B 的干净窗口训练）  %.6f" % mae_clean)
print("同一个模型在 95 个跨界样本上         %.6f（是前者的 %.1f 倍）"
      % (mae_cross, mae_cross / mae_clean))
print("混进 95 个跨界样本后 B 测试段        %.6f（%+.6f）"
      % (mae_mix, mae_mix - mae_clean))
print("跨界样本「目标内部」最大单步跳变 %.2f kW" % max(jump_t))
print("对照：B 全年最大单步跳变 %.2f kW | A 全年最大单步跳变 %.2f kW"
      % (np.abs(np.diff(S_B)).max(), np.abs(np.diff(S_A)).max()))

# ---- 验收 ----
assert abs(mae_clean - 63.174770) < 1e-4, f"B 测试段干净 MAE 实际是 {mae_clean}"
assert abs(mae_cross - 165.186264) < 1e-4, f"跨界样本 MAE 实际是 {mae_cross}"
assert mae_cross > 2 * mae_clean
assert abs(mae_mix - 63.208420) < 1e-4
assert abs(max(jump_t) - 349.35) < 1e-2, "跨界样本制造了全年最大的单步跳变"

## 任务 5：`DataLoader` 的形状与 `shuffle` 的合法边界（§6.2.1）

In [ ]:
ds_pv = WindowDataset([PV3], WINDOW, HORIZON)
print("光伏样本数 %d | 特征数 F=%d | 单个样本 x %s"
      % (len(ds_pv), PV3.shape[1], tuple(ds_pv[0][0].shape)))

with torch.random.fork_rng():
    torch.manual_seed(SEED)
    dl = torch.utils.data.DataLoader(ds_pv, batch_size=8, shuffle=True, drop_last=True)

    xb, yb = next(iter(dl))

idx_ds = torch.utils.data.TensorDataset(torch.arange(20))
dl_seq = torch.utils.data.DataLoader(idx_ds, batch_size=8)
order_seq = [int(v) for v in next(iter(dl_seq))[0]]        # shuffle=False：一定是 0..7

with torch.random.fork_rng():
    torch.manual_seed(SEED)
    dl_shuf = torch.utils.data.DataLoader(idx_ds, batch_size=8, shuffle=True)
    order_shuf = [int(v) for v in next(iter(dl_shuf))[0]]

print("一个 batch：x %s | y %s   ← (B, window, F) 与 (B, horizon)"
      % (tuple(xb.shape), tuple(yb.shape)))
print("%d 个样本 / batch=8 / drop_last=True → %d 个 batch（丢掉 %d 个）"
      % (len(ds_pv), len(dl), len(ds_pv) - len(dl) * 8))
print("shuffle=False 的第一个 batch 下标:", order_seq)
print("shuffle=True  的第一个 batch 下标:", order_shuf, "← 打乱了，但固定种子下可复现")

# ---- 验收 ----
assert len(ds_pv) == 8665 and len(dl) == 1083
assert tuple(xb.shape) == (8, 72, 3) and tuple(yb.shape) == (8, 24)
assert order_seq == list(range(8))
assert sorted(order_shuf) != order_seq, "shuffle=True 必须真的打乱了顺序"
assert len(set(order_shuf)) == 8

## 任务 6：手写 z-score / min-max 的 `fit` 与 `apply`（§6.3）

In [ ]:
def fit_zscore(x):
    return float(np.mean(x)), float(np.std(x))


def apply_zscore(x, mu, sd):
    return (x - mu) / sd


def fit_minmax(x):
    return float(x.min()), float(x.max())


def apply_minmax(x, lo, hi):
    return (x - lo) / (hi - lo)


probe = np.array([[1.0], [2.0], [3.0]])
z_hand = apply_zscore(probe, *fit_zscore(probe))
m_hand = apply_minmax(probe, *fit_minmax(probe))
print("手写 z-score →", z_hand.ravel(), "| sklearn StandardScaler →",
      StandardScaler().fit_transform(probe).ravel())
print("手写 min-max →", m_hand.ravel(), "| sklearn MinMaxScaler   →",
      MinMaxScaler().fit_transform(probe).ravel())

# ---- 验收 ----
assert np.allclose(z_hand, StandardScaler().fit_transform(probe))
assert np.allclose(m_hand, MinMaxScaler().fit_transform(probe))
assert np.allclose(m_hand.ravel(), [0.0, 0.5, 1.0])

## 任务 7：z-score —— 全量拟合比训练段拟合好多少？（§6.3.1）

In [ ]:
SEL_A = np.arange(len(S_A) - WINDOW - HORIZON + 1)
SPLIT_A = int(0.8 * len(S_A))
Xa_tr, Ya_tr = make_xy(S_A, SEL_A[:SPLIT_A - WINDOW - HORIZON + 1])
Xa_te, Ya_te = make_xy(S_A, SEL_A[SPLIT_A:])
Xa_all = np.concatenate([Xa_tr, Xa_te])
print("A 台区：训练 %d 个窗口 / 测试 %d 个窗口" % (len(Xa_tr), len(Xa_te)))

mu_tr, sd_tr = fit_zscore(Xa_tr)
mae_z_tr = float(mean_absolute_error(
    Ya_te.ravel(),
    Ridge(alpha=1.0).fit(apply_zscore(Xa_tr, mu_tr, sd_tr), Ya_tr)
    .predict(apply_zscore(Xa_te, mu_tr, sd_tr)).ravel()))

mu_all, sd_all = fit_zscore(Xa_all)
mae_z_all = float(mean_absolute_error(
    Ya_te.ravel(),
    Ridge(alpha=1.0).fit(apply_zscore(Xa_tr, mu_all, sd_all), Ya_tr)
    .predict(apply_zscore(Xa_te, mu_all, sd_all)).ravel()))

print("z-score：训练段拟合 %.6f | 全量拟合 %.6f | 差 %+.8f"
      % (mae_z_tr, mae_z_all, mae_z_all - mae_z_tr))
print("训练段 mean/std %.6f / %.6f | 全量 mean/std %.6f / %.6f"
      % (mu_tr, sd_tr, mu_all, sd_all))
print("测试段 mean %.6f（比训练段高 %.2f%%）"
      % (S_A[SPLIT_A:].mean(), 100 * (S_A[SPLIT_A:].mean() / S_A[:SPLIT_A].mean() - 1)))

# ---- 验收 ----
assert abs(mae_z_tr - 17.480038) < 1e-5, f"训练段拟合 MAE 实际是 {mae_z_tr}"
assert abs(mae_z_all - 17.480042) < 1e-5
assert abs(mae_z_all - mae_z_tr) < 1e-4, "这是负结果：全量拟合在本数据上几乎没影响"

## 任务 8：min-max —— 负结果最彻底的那一个（§6.3.1）

In [ ]:
lo_tr, hi_tr = fit_minmax(Xa_tr)
mae_m_tr = float(mean_absolute_error(
    Ya_te.ravel(),
    Ridge(alpha=1.0).fit(apply_minmax(Xa_tr, lo_tr, hi_tr), Ya_tr)
    .predict(apply_minmax(Xa_te, lo_tr, hi_tr)).ravel()))

lo_all, hi_all = fit_minmax(Xa_all)
mae_m_all = float(mean_absolute_error(
    Ya_te.ravel(),
    Ridge(alpha=1.0).fit(apply_minmax(Xa_tr, lo_all, hi_all), Ya_tr)
    .predict(apply_minmax(Xa_te, lo_all, hi_all)).ravel()))

print("min-max：训练段拟合 %.6f | 全量拟合 %.6f | 差 %+.8f"
      % (mae_m_tr, mae_m_all, mae_m_all - mae_m_tr))
print("训练段 min/max %.4f / %.4f | 全量 min/max %.4f / %.4f"
      % (lo_tr, hi_tr, lo_all, hi_all))
print("训练段与全量的极值一模一样？%s  ← min-max 的参数完全没变，MAE 自然一分不差"
      % ((lo_tr, hi_tr) == (lo_all, hi_all)))
print("四宫格（kW）：z-score 训练段 %.6f / 全量 %.6f | min-max 训练段 %.6f / 全量 %.6f"
      % (mae_z_tr, mae_z_all, mae_m_tr, mae_m_all))

# ---- 验收 ----
assert abs(mae_m_tr - 17.464098) < 1e-5, f"min-max 训练段拟合 MAE 实际是 {mae_m_tr}"
assert (lo_tr, hi_tr) == (lo_all, hi_all), "A 台区的极值本来就落在训练段里"
assert abs(mae_m_all - mae_m_tr) < 1e-12, "min-max 的负结果比 z-score 还彻底：一分不差"
assert abs(mae_z_all - mae_z_tr) < 0.001

## 任务 9：手写 `TimeSeriesSplit`（§6.4）

In [ ]:
def ts_split(n, n_splits, gap=0):
    """手写 TimeSeriesSplit：返回 [(train_idx, valid_idx), ...]。

    第 k 折的**验证段**起点固定在 k * fold（与 sklearn 一致），
    训练段则从 0 开始一直铺到「验证段起点 - gap」。
    """
    fold = n // (n_splits + 1)
    out = []
    for k in range(1, n_splits + 1):
        va_lo = k * fold
        tr_end = va_lo - gap
        va_hi = n if k == n_splits else va_lo + fold
        out.append((np.arange(max(tr_end, 0)), np.arange(va_lo, va_hi)))
    return out


folds_0 = ts_split(8760, 4, 0)
print("每折的 fold 长度 = 8760 // 5 = %d（8760 / 5 正好整除）" % (8760 // 5))
for k, (tr, va) in enumerate(ts_split(8760, 4, 0), start=1):
    print("  第 %d 折：训练 [0, %4d) 共 %4d | 验证 [%4d, %4d) 共 %d"
          % (k, tr[-1] + 1, len(tr), va[0], va[-1] + 1, len(va)))

# ---- 验收 ----
assert len(folds_0) == 4
assert [len(tr) for tr, _ in folds_0] == [1752, 3504, 5256, 7008]
assert [len(va) for _, va in folds_0] == [1752, 1752, 1752, 1752]
assert [int(tr[-1]) + 1 for tr, _ in folds_0] == [1752, 3504, 5256, 7008]
assert [int(va[0]) for _, va in folds_0] == [1752, 3504, 5256, 7008]

## 任务 10：与 sklearn 逐折对账 + 看清 `gap` 削掉了什么（§6.4.1）

In [ ]:
n = 8760
checks = {}
for g in (0, 24, 48):
    mine, skl = ts_split(n, 4, g), list(TimeSeriesSplit(n_splits=4, gap=g).split(np.arange(n)))
    same_tr = [np.array_equal(a, b) for (a, _), (b, _) in zip(mine, skl)]
    same_va = [np.array_equal(c, d) for (_, c), (_, d) in zip(mine, skl)]
    checks[g] = all(same_tr) and all(same_va)

edges0 = [(int(a[-1]) + 1, int(c[0]), int(c[-1]) + 1) for a, c in ts_split(n, 4, 0)]
edges24 = [(int(a[-1]) + 1, int(c[0]), int(c[-1]) + 1) for a, c in ts_split(n, 4, 24)]

print("手写 vs sklearn 对账：", {g: ("一致" if ok else "不一致") for g, ok in checks.items()})
print("gap=0  折边界 (train_end, valid_lo, valid_hi):", edges0)
print("gap=24 折边界 (train_end, valid_lo, valid_hi):", edges24)
print("对比第 1 折：训练段末端 %d → %d（少喂 %d 个点），验证段起点 %d 没变"
      % (edges0[0][0], edges24[0][0], edges0[0][0] - edges24[0][0], edges0[0][1]))

# ---- 验收 ----
assert all(checks.values()), "手写实现必须与 sklearn 逐折一致"
assert edges24[0][0] == 1728 and edges24[0][1] == 1752 and edges24[0][2] == 3504
assert edges0 == [(1752, 1752, 3504), (3504, 3504, 5256), (5256, 5256, 7008), (7008, 7008, 8760)]
assert [a - b for (a, _, _), (b, _, _) in zip(edges0, edges24)] == [24, 24, 24, 24]

## 任务 11：滑窗 vs 扩展窗，4 折实测（§6.5.1）

In [ ]:
FOLDS = ts_split(len(S_A), 4, 0)
NMAX = len(S_A) - WINDOW - HORIZON + 1
SLIDE_LEN = 1752
fold_mae = {"expanding": [], "sliding": []}

for name in ("expanding", "sliding"):
    for tr, va in FOLDS:
        tr_end = int(tr[-1]) + 1
        v_lo, v_hi = int(va[0]), min(int(va[-1]) + 1, NMAX)
        Xv, Yv = make_xy(S_A, np.arange(v_lo, v_hi))
        lo_i = 0 if name == "expanding" else max(0, tr_end - SLIDE_LEN)
        hi_i = min(tr_end - WINDOW - HORIZON + 1, NMAX)
        Xt, Yt = make_xy(S_A, np.arange(lo_i, hi_i))
        fold_mae[name].append((hi_i - lo_i, ridge_mae(Xt, Yt, Xv, Yv)))
    sizes = [c for c, _ in fold_mae[name]]
    maes = [round(m, 4) for _, m in fold_mae[name]]
    print("%-9s 训练样本数 %s | 各折 MAE %s | 折均 %.6f"
          % (name, sizes, maes, sum(m for _, m in fold_mae[name]) / len(fold_mae[name])))

exp_mean = sum(m for _, m in fold_mae["expanding"]) / 4
sld_mean = sum(m for _, m in fold_mae["sliding"]) / 4
print("折均 MAE：扩展窗 %.6f vs 滑窗 %.6f（差 %+.6f）" % (exp_mean, sld_mean, sld_mean - exp_mean))

# ---- 验收 ----
assert [c for c, _ in fold_mae["expanding"]] == [1657, 3409, 5161, 6913]
assert [c for c, _ in fold_mae["sliding"]] == [1657, 1657, 1657, 1657]
assert abs(fold_mae["expanding"][2][1] - 16.926867) < 1e-4
assert abs(sld_mean - exp_mean - 0.741726) < 1e-3, "扩展窗 4 折平均更好"

## 任务 12：两张图（折切分示意 + 折 MAE 对照）（§6.5）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))

# ---- 左图：TimeSeriesSplit 4 折 + gap=24 的示意 ----
for k, (tr, va) in enumerate(ts_split(len(S_A), 4, 24), start=1):
    axes[0].barh(k, len(tr), left=0, height=0.55, color="tab:blue", alpha=0.85)
    axes[0].barh(k, 24, left=len(tr), height=0.55, color="crimson", alpha=0.6)
    axes[0].barh(k, len(va), left=int(va[0]), height=0.55, color="tab:orange", alpha=0.85)
axes[0].plot([], [], "s", color="tab:blue", label="训练段")
axes[0].plot([], [], "s", color="crimson", alpha=0.6, label="gap=24")
axes[0].plot([], [], "s", color="tab:orange", label="验证段")
axes[0].set_xlabel("时间（小时）")
axes[0].set_ylabel("第 k 折")
axes[0].set_yticks([1, 2, 3, 4])
axes[0].set_title("TimeSeriesSplit(4, gap=24)：验证段位置固定，训练段整体后退 gap")
axes[0].legend(fontsize=9, loc="lower right")

exp_mae = [m for _, m in fold_mae["expanding"]]
sld_mae = [m for _, m in fold_mae["sliding"]]
exp_n = [c for c, _ in fold_mae["expanding"]]
sld_n = [c for c, _ in fold_mae["sliding"]]
axes[1].plot(np.arange(1, 5), exp_mae, "-o", label="扩展窗 expanding（样本 1657→6913）")
axes[1].plot(np.arange(1, 5), sld_mae, "--s", label="滑窗 sliding（样本恒为 1657）")

axes[1].axhline(exp_mean, ls=":", color="tab:blue", lw=1)
axes[1].axhline(sld_mean, ls=":", color="tab:orange", lw=1)
axes[1].set_xlabel("第 k 折")
axes[1].set_ylabel("验证折 MAE（kW）")
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_title("滑窗 vs 扩展窗：4 折 MAE（折均 %.4f vs %.4f）" % (exp_mean, sld_mean))
axes[1].legend(fontsize=9)
plt.tight_layout()

print("左图：4 折 + gap=24 的区间示意；右图：滑窗 vs 扩展窗的折 MAE")
print("注意第 1 折两条线重合（数据量都是 1657）—— 差异从第 2 折开始")

# ---- 验收 ----
assert len(exp_mae) == 4 and abs(exp_mae[0] - sld_mae[0]) < 1e-12, "第 1 折两者必然相同"
assert exp_mae[1] < sld_mae[1], "第 2 折起扩展窗数据更多，MAE 更低"

## 自查清单（能不看笔记答出来才算过）

- [ ] 12 个代码块的 assert 全部通过
- [ ] 把 A / B 直接 `np.concatenate` 再切窗，会多出多少个样本？为什么？
- [ ] 「输入窗口跨边界」和「目标跨边界」各有多少个？
- [ ] 干净模型在跨界样本上的 MAE 是测试段的几倍？
- [ ] 跨界样本的目标里最大的单步跳变是多少？比真实数据大多少？
- [ ] `shuffle=True` 在时序上什么时候可以开、什么时候绝对不能开？
- [ ] `drop_last=True` 该给训练集还是验证集？
- [ ] `WindowDataset` 的索引表为什么存 `(第几条序列, 起点)` 而不是只存起点？
- [ ] z-score 与 min-max 的公式分别是什么？哪个对极值敏感？
- [ ] 全量拟合 vs 训练段拟合，z-score 差多少？min-max 差多少？为什么 min-max 是 0？
- [ ] `gap` 参数改变的是训练段还是验证段？本章最小的合法 `gap` 是多少？
- [ ] 滑窗与扩展窗，第几折开始出现差异？折均差多少？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| A / B / PV 形状 | (8760,) / (8760,) / **(8760, 3)** |
| A / B 标准差比 | **3.8943** |
| 光伏实际功率清洗后 min | **0.0**（负值被 clip） |
| 分组切窗样本数 | **17330**（8665 × 2） |
| concat 切窗样本数 | **17425** |
| **跨界样本数** | **95**（输入跨界 71 / 目标跨界 24） |
| `stride=4` 的样本数 | 4334（每条 2167） |
| `DataLoader(batch=8, drop_last=True)` | **1083** 个 batch，形状 (8, 72, 3) / (8, 24) |
| B 测试段 MAE（干净训练） | **63.174770** |
| 干净模型在 95 个跨界样本上的 MAE | **165.186264**（**2.6 倍**） |
| 混入 95 个跨界样本后的 B 测试段 MAE | 63.208420（**+0.033649**） |
| 跨界样本目标内最大单步跳变 | **349.35 kW**（A 142.34 / B 303.38） |
| z-score：训练段 / 全量 | **17.480038 / 17.480042**（差 **+3.81e-6**） |
| min-max：训练段 / 全量 | 17.464098 / 17.464098（差 **0**） |
| z-score 参数：训练段 → 全量 | 均值 454.520355 → 459.835022（+1.17%）；std 72.797569 → 72.121338 |
| A 训练段 min/max vs 全量 | 275.3900/739.9100 **完全相同** |
| `ts_split(8760, 4, 0)` 折边界 | (1752,1752,3504) (3504,3504,5256) (5256,5256,7008) (7008,7008,8760) |
| `gap=24` 第 1 折 | train_end **1728** / valid [1752, 3504) |
| 手写 vs sklearn（gap=0/24/48） | **全部一致** |
| 扩展窗 4 折 MAE | 19.718073 / 17.931772 / 16.926867 / 17.480038，折均 **18.014187** |
| 滑窗 4 折 MAE | 19.718073 / 19.012787 / 17.426353 / 18.866442，折均 **18.755914** |

> **三条纪律**：切窗先分组；切分不打乱、训练集内打乱；缩放只在训练段 `fit`。